# Data Profiling & Quality Risk Analysis
## Workshop-2: Building a Reliable Batch Data Pipeline
**Curso:** ETL (G01) — Universidad Autónoma de Occidente (UAO)  
**Objetivo:** Recolección sistemática de evidencia sobre las fuentes de datos (Spotify y Grammy Awards) para justificar técnicamente las reglas de calidad de datos (Great Expectations) y las transformaciones de ingeniería.

---
### Categorías Mínimas de Análisis:
1. **Estructura:** Esquemas, columnas, tipos de datos, conteo de filas.
2. **Completitud:** Nulos absolutos y relativos en contexto.
3. **Unicidad y Duplicación:** Claves candidatas y análisis del grano.
4. **Contenido Categórico:** Cardinalidad, variaciones, etiquetas.
5. **Contenido Numérico:** Rangos, valores anómalos/imposibles, distribuciones.
6. **Contenido Temporal:** Cobertura, parseabilidad de fechas y granularidad.
7. **Consistencia Cruzada:** Estrategia de emparejamiento entre Spotify y Grammys.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

print("Librerías importadas exitosamente.")


## 1. Carga de Datos Crudos

In [ ]:
# Carga de datasets
df_spotify = pd.read_csv("../spotify_dataset.csv")
df_grammy = pd.read_csv("../the_grammy_awards.csv")

print(f"Dimensiones Spotify: {df_spotify.shape[0]:,} filas, {df_spotify.shape[1]} columnas")
print(f"Dimensiones Grammys: {df_grammy.shape[0]:,} filas, {df_grammy.shape[1]} columnas")


## 2. Análisis de Estructura y Tipos de Datos

In [ ]:
print("=== ESTRUCTURA SPOTIFY ===")
display(df_spotify.dtypes.to_frame("Tipo de Dato"))

print("\n=== ESTRUCTURA GRAMMYS ===")
display(df_grammy.dtypes.to_frame("Tipo de Dato"))


## 3. Análisis de Completitud (Valores Faltantes y Nulos)

In [ ]:
# Nulos en Spotify
null_spotify = pd.DataFrame({
    "Nulos": df_spotify.isnull().sum(),
    "% Nulos": (df_spotify.isnull().sum() / len(df_spotify)) * 100
})
print("Nulos en Spotify:")
display(null_spotify[null_spotify["Nulos"] > 0])

# Nulos en Grammys
null_grammy = pd.DataFrame({
    "Nulos": df_grammy.isnull().sum(),
    "% Nulos": (df_grammy.isnull().sum() / len(df_grammy)) * 100
})
print("Nulos en Grammys:")
display(null_grammy[null_grammy["Nulos"] > 0])


## 4. Análisis de Unicidad, Claves Candidatas y Grano

In [ ]:
print("=== UNICIDAD SPOTIFY ===")
total_rows = len(df_spotify)
unique_tracks = df_spotify["track_id"].nunique()
dup_track_ids = df_spotify["track_id"].duplicated().sum()
dup_track_genre = df_spotify.duplicated(subset=["track_id", "track_genre"]).sum()

print(f"Total registros: {total_rows:,}")
print(f"IDs únicos (track_id): {unique_tracks:,}")
print(f"Filas con track_id duplicado: {dup_track_ids:,} (canción aparece en múltiples géneros)")
print(f"Duplicados exactos (track_id, track_genre): {dup_track_genre}")

print("\n=== UNICIDAD GRAMMYS ===")
dup_grammy_natural = df_grammy.duplicated(subset=["year", "category", "nominee"]).sum()
print(f"Duplicados en clave natural (year, category, nominee): {dup_grammy_natural}")


## 5. Análisis de Contenido Categórico

In [ ]:
print(f"Total géneros musicales únicos en Spotify: {df_spotify['track_genre'].nunique()}")
print(f"Top 10 géneros con más pistas:\n{df_spotify['track_genre'].value_counts().head(10)}")

print(f"\nTotal categorías en Grammys: {df_grammy['category'].nunique()}")
print(f"Distribución de la columna winner en Grammys:\n{df_grammy['winner'].value_counts()}")


## 6. Análisis de Contenido Numérico y Valores Extremos

In [ ]:
num_cols = ["popularity", "duration_ms", "danceability", "energy", "loudness", "valence", "tempo"]
display(df_spotify[num_cols].describe().round(3))

# Detección de valores imposibles
zero_duration = (df_spotify["duration_ms"] == 0).sum()
zero_tempo = (df_spotify["tempo"] == 0).sum()
print(f"Pistas con duración igual a 0 ms: {zero_duration}")
print(f"Pistas con tempo igual a 0 BPM: {zero_tempo}")


## 7. Análisis de Contenido Temporal

In [ ]:
print(f"Rango de años en Grammys: {df_grammy['year'].min()} a {df_grammy['year'].max()}")
print(f"Total años cubiertos: {df_grammy['year'].nunique()}")

# Parsing de fechas en formato ISO con timezone
df_grammy["pub_parsed"] = pd.to_datetime(df_grammy["published_at"], utc=True, errors="coerce")
invalid_dates = df_grammy["pub_parsed"].isnull().sum()
print(f"Fechas no parseables en published_at: {invalid_dates}")


## 8. Análisis de Consistencia Cruzada e Integración (Matching Contract)

In [ ]:
# Prueba de cruce por título de pista
spotify_titles = df_spotify["track_name"].dropna().str.lower().str.strip()
grammy_nominees = df_grammy["nominee"].dropna().str.lower().str.strip()

exact_title_matches = len(set(spotify_titles).intersection(set(grammy_nominees)))
print(f"Coincidencias directas por título de canción: {exact_title_matches:,}")

# Ejemplo de variación de artistas
print("\nEjemplo de formato de artistas:")
print("Spotify:", df_spotify["artists"].iloc[2])
print("Grammys:", df_grammy[df_grammy["artist"].notnull()]["artist"].iloc[0])


## 9. Síntesis de Evidencia y Matriz de Riesgos de Calidad

| Dataset / Atributo | Evidencia Observada en Perfilamiento | Riesgo Potencial de Calidad | Severidad Justificada | Requisito Relacionado |
| :--- | :--- | :--- | :--- | :--- |
| `spotify.track_id` | 450 filas duplicadas exactamente para la misma tupla `(track_id, track_genre)`. | Duplicación no intencional y distorsión de agregaciones en el DW. | **Critical** (Compuerta Raw) | RQ-01, RQ-02 |
| `spotify.duration_ms` | Mínimo observado = 0 ms (pistas fantasma/corruptas). | Métricas de duración promedio y análisis musical inválidos. | **Critical** (Compuerta Raw) | RQ-03 |
| `spotify.audio_features` | Valores esperados en escala [0.0, 1.0]. | Distorsión de correlaciones acústicas si existen valores fuera de rango. | **Critical** (Compuerta Raw) | RQ-01 |
| `spotify.popularity` | Valores entre 0 y 100. 1 fila con nulos en metadatos principales. | Nulos rompen la integridad referencial de dimensiones. | **Critical** (Compuerta Raw) | RQ-02 |
| `grammy.year` | Rango [1958, 2019]. Años fuera de rango o nulos corrompen la dimensión tiempo. | Agrupaciones históricas inválidas en el Data Warehouse. | **Critical** (Compuerta Raw) | RQ-01, RQ-03 |
| `grammy.nominee` | 6 filas con `nominee` nulo en la fuente. | Imposibilidad absoluta de cruce con Spotify. | **Warning** (Compuerta Raw) | RQ-01, RQ-02 |
| `integrated.unmatched` | Muchas pistas de Spotify no tienen Grammy (comportamiento esperado del negocio). | Falso positivo si se espera 100% match; debe modelarse con flag `is_grammy_winner`. | **Informational** (Compuerta Prepared) | RQ-01, RQ-02 |
| `prepared.audio_bounds` | Post-transformación las características deben conservar escala $[0.0, 1.0]$. | Datos preparados aptos para carga dimensional. | **Critical** (Compuerta Prepared) | RQ-01 |
